# z713 — LightGBM a nivel producto, segmentado por los top-13 clientes

## Por qué producto y no par

Medido sobre los mismos folds, con Optuna y semillas en ambos:

| | 201902 (predice el LB) | meses que Optuna **no** vio |
|---|---|---|
| **producto** | **0,1932** | **0,2729** |
| par ⟨cliente, producto⟩ (z711) | 0,2708 → LB 0,275 | 0,3289 |

Un 29 % mejor en el fold que anticipó el leaderboard, y 17 % en los meses limpios. No es
tuning: es el nivel de agregación. El par no aporta información nueva sobre lo que hay que
predecir —son los mismos 36 meses partidos en 736 k series con 80 % de ceros— y sí aporta
ruido que la métrica encima perdona, porque los errores entre clientes del mismo producto se
cancelan antes de puntuar.

Y hay un efecto secundario que importa: **producto-mes son 31 522 filas** contra 17 M. Con
`segmentar` y `comparar_sin_segmentar` prendidos, cada trial entrena 2 modelos × 2 folds, así
que 300 trials son unas **2–3 h** en 8 núcleos (menos en la VM). Con el par-nivel, esas mismas
horas alcanzaban para 30 trials. Si querés la vuelta rápida, `n_trials = 60` ya deja el
resultado a menos de 1 % del de 300 y corre en media hora.

## La segmentación por top-13

Los 13 clientes más grandes son el **50,8 % del tonelaje**. En vez de un modelo por producto, se
arman **dos series por producto** —lo que le compran los 13 grandes y lo que le compra el
resto—, se entrena un modelo para cada mundo y se **suman las predicciones**. Los dos regímenes
no se contaminan: los grandes compran todos los meses y con volumen estable, la cola compra
salteado.

Medido, gana en 4 de 5 folds (201902: 0,1948 → 0,1896). La diferencia media pareada es
**−0,0040 ± 0,0026**, o sea direccionalmente positiva pero por debajo de dos errores estándar.
Es una mejora chica y barata, no un cambio de régimen — y como acá el cómputo sobra, el
notebook entrena **las dos versiones siempre** y muestra la comparación pareada fold por fold.

## El target va crudo (y por qué acá sí)

Medido: crudo 0,1932 contra escalado `tn/s` 0,2066 en 201902, y 0,2462 contra 0,2562 en el
promedio de los cinco folds.

Escalar sirve cuando las series difieren cinco órdenes de magnitud, como los 736 k pares —ahí
sin escalar el modelo sólo aprende de las grandes—. A nivel producto son 1233 series mucho más
parejas, y como WAPE pondera por tonelaje, **el target crudo ya está alineado con la métrica**.
Escalar mete una división que después hay que deshacer, y cada paso agrega error.

`escalar = True` deja el target en `tn(t+2)/s(t)` con `s` = media expandida causal, por si la
consigna manda. Funciona, sólo que un poco peor.

## La ley inviolable

La clase es lo único que mira el futuro. Todo el FE sale de `<= t`, y la celda 4 lo verifica
recalculándolo sobre historia truncada: si una feature cambia, **aborta**. Los folds entrenan
con `periodo <= corte − 2` y evalúan en `corte`, que es el horizonte real del deploy.

**Sin multiplicador**: 7 de 7 veces perdió en el leaderboard. Pero la celda 7 sí **reporta** el
ratio `real / predicho` fold por fold: en el smoke test el modelo se quedaba corto en la mediana
de los meses, con una dispersión alta entre ellos (0,98 a 1,33). Un multiplicador global no está
determinado con esa dispersión, y el total de la inferencia contra la banda de febrero te dice
si el problema persiste. Mirá los dos números antes de subir.

In [ ]:
%pip install -q polars lightgbm optuna pandas numpy pyarrow

In [ ]:
# ruff: noqa: E402
import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAY_OPTUNA = True
except Exception:
    HAY_OPTUNA = False

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/.drive")
    os.makedirs("/content/buckets", exist_ok=True)
    if not os.path.islink("/content/buckets/b1"):
        os.symlink("/content/.drive/My Drive/labo3", "/content/buckets/b1")
    os.environ["LABO3_BUCKET"] = "/content/buckets/b1"


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    for c in ("/home/jupyter/buckets/b1", "/content/buckets/b1", "~/buckets/b1"):
        p = Path(c).expanduser()
        if p.exists():
            return p
    p = Path.cwd() / "bucket"; p.mkdir(parents=True, exist_ok=True); return p


BUCKET = resolver_bucket()
N_CORES = os.cpu_count() or 8
print(f"bucket: {BUCKET} | cores: {N_CORES} | optuna: {HAY_OPTUNA}")

## 1 — Palancas

In [ ]:
PARAM = {
    "experimento": "z713_producto_top13",
    "kaggle_competition": "labo-iii-2026-ba",

    "horizonte": 2,
    "periodo_inferencia": 201912,
    "n_top": 13,                 # clientes grandes que van en su propio modelo
    "segmentar": True,           # entrena top13 y resto por separado, y suma
    "comparar_sin_segmentar": True,   # entrena tambien el modelo unico y los compara (barato)
    "escalar": False,            # False = target crudo tn(t+2). True = tn(t+2)/s(t)
    "piso_escala": 1e-3,

    "features": ["lag1", "lag2", "lag3", "lag12", "rm3", "rm12", "rs3",
                 "nivel", "tend", "v12", "logs", "mes", "cat3", "brand"],
    "categoricas": ["cat3", "brand"],

    # tunear en febreros; los otros meses solo se reportan
    "anclas_optuna": [201802, 201902],
    "anclas_reporte": [201802, 201810, 201902, 201906, 201910],

    "n_trials": 300,             # a 31k filas esto son minutos
    "max_bin": 1023,
    "semillas": [102191, 314159, 777773, 116269, 241511, 500009, 660013,
                 880007, 979103, 859553, 769339, 705389, 575711, 495787, 438029],

    "banda_nivel": [28400, 30600],
    "submit": True,
    "sufijo": "",
}

DIR_RAW = BUCKET / "datasets"
DIR_EXP = BUCKET / "exp" / (PARAM["experimento"] + PARAM["sufijo"])
DIR_OUT = DIR_EXP / "submits"
for d in (DIR_RAW, DIR_EXP, DIR_OUT):
    d.mkdir(parents=True, exist_ok=True)
H = PARAM["horizonte"]
FEATS = PARAM["features"]
CAT = PARAM["categoricas"]
print(f"experimento: {DIR_EXP}")
print(f"features ({len(FEATS)}): {FEATS}")
print(f"segmentar top-{PARAM['n_top']}: {PARAM['segmentar']} | target: "
      f"{'escalado tn/s' if PARAM['escalar'] else 'CRUDO'}")

## 2 — Datos y series

Se agrega a producto-mes. Con `segmentar`, cada producto se parte en dos series: los `n_top`
clientes más grandes y el resto. La grilla es densa dentro de la vida de cada serie, forzada
hasta el periodo de inferencia para los 780 — si no, un producto cuya serie de top-13 se apagó
antes no tendría fila para predecir y aportaría 0 por construcción.

In [ ]:
def descargar(a: str):
    dst = DIR_RAW / a
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{a}"
    subprocess.run(["wget", "-q", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prods = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
         .select(["product_id", "cat3", "brand"]).unique(subset=["product_id"]))
TARGET = set(pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt",
                         separator="\t")["product_id"].to_list())
CAL = sorted(sell["periodo"].unique().to_list())

vol_cli = sell.group_by("customer_id").agg(pl.col("tn").sum()).sort("tn", descending=True)
TOP = vol_cli["customer_id"].head(PARAM["n_top"]).to_list()
share = 100 * sell.filter(pl.col("customer_id").is_in(TOP))["tn"].sum() / sell["tn"].sum()
print(f"top-{PARAM['n_top']} clientes = {share:.1f}% del tonelaje")
sell = sell.with_columns(pl.col("customer_id").is_in(TOP).cast(pl.Int8).alias("top"))


def grilla(df: pl.DataFrame) -> pl.DataFrame:
    """Grid denso producto-mes dentro de la vida de la serie, extendido a la inferencia."""
    vida = df.group_by("product_id").agg([pl.col("periodo").min().alias("p0"),
                                          pl.col("periodo").max().alias("p1")])
    vida = vida.with_columns(
        pl.when(pl.col("product_id").is_in(list(TARGET)))
        .then(pl.max_horizontal(pl.col("p1"), pl.lit(PARAM["periodo_inferencia"])))
        .otherwise(pl.col("p1")).alias("p1"))
    return (vida.join(pl.DataFrame({"periodo": CAL}), how="cross")
            .filter((pl.col("periodo") >= pl.col("p0")) & (pl.col("periodo") <= pl.col("p1")))
            .select(["product_id", "periodo"])
            .join(df, on=["product_id", "periodo"], how="left")
            .with_columns(pl.col("tn").fill_null(0.0))
            .join(prods, on="product_id", how="left")
            .sort(["product_id", "periodo"]))

## 3 — Feature engineering

Catorce features, todas relativas a la escala del producto o adimensionales, salvo `logs` —el
log de `s(t)`— que es la que le da la magnitud. El `.over(product_id)` va **al final** de la
cadena: adentro del `shift` la ventana móvil corre sobre la columna entera y se derrama de la
serie anterior.

In [ ]:
def fe(g: pl.DataFrame) -> pl.DataFrame:
    q = "product_id"
    tn = pl.col("tn")
    g = g.with_columns((tn.cum_sum().over(q) / pl.int_range(1, pl.len() + 1).over(q)).alias("s"))
    e = pl.max_horizontal(pl.col("s"), pl.lit(PARAM["piso_escala"]))
    g = g.with_columns([
        (tn.shift(1).over(q) / e).alias("lag1"), (tn.shift(2).over(q) / e).alias("lag2"),
        (tn.shift(3).over(q) / e).alias("lag3"), (tn.shift(12).over(q) / e).alias("lag12"),
        (tn.shift(1).rolling_mean(3, min_samples=1).over(q) / e).alias("rm3"),
        (tn.shift(1).rolling_mean(12, min_samples=1).over(q) / e).alias("rm12"),
        (tn.shift(1).rolling_std(3, min_samples=2).over(q) / e).alias("rs3"),
        (tn / e).alias("nivel"),
        ((tn > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).shift(1).over(q)).alias("v12"),
        (pl.col("periodo") % 100).alias("mes"),
        pl.col("s").log1p().alias("logs"),
    ])
    return g.with_columns([(pl.col("rm3") / (pl.col("rm12") + 1e-6)).alias("tend"),
                           tn.shift(-H).over(q).alias("y")])


pp = sell.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum())
SERIES = {"unico": fe(grilla(pp))}
if PARAM["segmentar"]:
    ppt = sell.group_by(["product_id", "top", "periodo"]).agg(pl.col("tn").sum())
    for t, nom in ((1, "top13"), (0, "resto")):
        SERIES[nom] = fe(grilla(ppt.filter(pl.col("top") == t).drop("top")))

COD = {c: {v: i for i, v in enumerate(sorted(SERIES["unico"][c].unique().drop_nulls().to_list()))}
       for c in CAT}
DF = {}
for k, v in SERIES.items():
    d = v.to_pandas()
    for c in CAT:
        d[c] = d[c].map(COD[c]).fillna(-1).astype("int32")
    DF[k] = d
    print(f"  {k:6s}: {len(d):,} filas | {d.product_id.nunique()} productos")

## 4 — Test de causalidad (aborta si falla)

In [ ]:
def test_causalidad(T: int = 201810, n: int = 200):
    ids = sorted(pp["product_id"].unique().to_list())[:n]
    sub = pp.filter(pl.col("product_id").is_in(ids))
    cols = [c for c in FEATS + ["s"] if c not in CAT]
    # Se comparan por CLAVE, no por posicion: `grilla` extiende la vida hasta el periodo de
    # inferencia para los 780, asi que la version truncada trae filas de mas y comparar
    # posicionalmente daria falsos positivos en todas las columnas.
    a = (fe(grilla(sub)).filter(pl.col("periodo") <= T)
         .select(["product_id", "periodo"] + cols))
    b = (fe(grilla(sub.filter(pl.col("periodo") <= T))).filter(pl.col("periodo") <= T)
         .select(["product_id", "periodo"] + cols))
    j = a.join(b, on=["product_id", "periodo"], how="inner", suffix="_t")
    assert j.height > 1000, f"muy pocas filas comparables ({j.height}): el test no prueba nada"
    malas = []
    for c in cols:
        x = np.nan_to_num(j[c].to_numpy().astype(float))
        y = np.nan_to_num(j[c + "_t"].to_numpy().astype(float))
        if not np.allclose(x, y, atol=1e-9):
            malas.append(c)
    if malas:
        raise AssertionError(f"FUGA DE FUTURO en: {malas}")
    print(f"test de causalidad OK: {len(cols)} columnas, {j.height:,} filas comparadas, "
          f"T={T}, {n} productos")


test_causalidad()
test_causalidad(T=201903)

## 5 — Motor

Un fold entrena con `periodo <= corte − 2` —la única ventana cuya clase ya ocurrió— y evalúa en
`periodo == corte`, cuya clase es el mes objetivo. Es el horizonte real del deploy.

In [ ]:
def pm(p: int, k: int) -> int:
    a, m = divmod(p, 100)
    t = a * 12 + (m - 1) - k
    return (t // 12) * 100 + (t % 12) + 1


def entrenar_predecir(d: pd.DataFrame, corte: int, params: dict, n: int, semillas: list):
    tr = d[(d.periodo <= pm(corte, H)) & d.y.notna()]
    ev = d[(d.periodo == corte) & d.product_id.isin(TARGET)].copy()
    if len(tr) == 0 or len(ev) == 0:
        return None
    s_tr = np.maximum(tr.s.values, PARAM["piso_escala"])
    s_ev = np.maximum(ev.s.values, PARAM["piso_escala"])
    lab = tr.y.values / s_tr if PARAM["escalar"] else tr.y.values
    ds = lgb.Dataset(tr[FEATS].values.astype(np.float32), label=lab, feature_name=FEATS,
                     categorical_feature=CAT,
                     params={"max_bin": PARAM["max_bin"], "verbosity": -1,
                             "feature_pre_filter": False})
    ds.construct()
    acc = np.zeros(len(ev))
    for sem in semillas:
        b = lgb.train({**params, "seed": sem, "verbosity": -1, "num_threads": N_CORES}, ds, n)
        p = b.predict(ev[FEATS].values.astype(np.float32))
        acc += np.maximum(p * s_ev if PARAM["escalar"] else p, 0.0)
    ev["h"] = acc / len(semillas)
    return ev[["product_id", "periodo", "y", "h"]]


def prediccion(modo: str, corte: int, params: dict, n: int, semillas: list) -> pd.Series:
    """modo='unico' o 'segmentado' (suma de top13 y resto)."""
    if modo == "unico":
        r = entrenar_predecir(DF["unico"], corte, params, n, semillas)
        return r.set_index("product_id")["h"]
    tot = None
    for k in ("top13", "resto"):
        r = entrenar_predecir(DF[k], corte, params, n, semillas)
        if r is None:
            continue
        h = r.set_index("product_id")["h"]
        tot = h if tot is None else tot.add(h, fill_value=0.0)
    return tot


def wape(real: pd.Series, pred: pd.Series) -> float:
    i = real.dropna().index
    p = pred.reindex(i).fillna(0.0)
    return float(np.abs(real[i] - p).sum() / real[i].sum())


REAL = {a: (DF["unico"].query("periodo == @pm(@a, @H) and product_id in @TARGET")
            .set_index("product_id")["y"]) for a in PARAM["anclas_reporte"]}

## 6 — Optuna

Se tunea sobre los folds de **febrero**, que es el mes objetivo. Los otros meses del reporte
quedan limpios: sirven para ver si generaliza, no para elegir. Con 31 k filas, 300 trials son
minutos — el cómputo se gasta acá, que es donde rinde.

In [ ]:
MODO = "segmentado" if PARAM["segmentar"] else "unico"


def espacio(t) -> dict:
    return {"objective": "tweedie",
            "tweedie_variance_power": t.suggest_float("tvp", 1.1, 1.6),
            "learning_rate": t.suggest_float("lr", 0.005, 0.2, log=True),
            "num_leaves": t.suggest_int("nl", 8, 128),
            "min_data_in_leaf": t.suggest_int("md", 5, 120),
            "feature_fraction": t.suggest_float("ff", 0.5, 1.0),
            "bagging_fraction": t.suggest_float("bf", 0.6, 1.0), "bagging_freq": 1,
            "lambda_l1": t.suggest_float("l1", 1e-4, 10, log=True),
            "lambda_l2": t.suggest_float("l2", 1e-4, 10, log=True),
            "max_bin": PARAM["max_bin"]}


CK = DIR_EXP / "mejores.json"
if CK.exists():
    MEJOR = json.loads(CK.read_text())
    print("hiperparametros desde checkpoint")
else:
    import time
    t0 = time.time()
    if HAY_OPTUNA:
        st = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=102191))

        def obj(t):
            p = espacio(t); n = t.suggest_int("n", 100, 1200)
            return float(np.mean([wape(REAL[a], prediccion(MODO, pm(a, H), p, n, [102191]))
                                  for a in PARAM["anclas_optuna"]]))

        st.optimize(obj, n_trials=PARAM["n_trials"], show_progress_bar=False)
        bp = st.best_params
        MEJOR = {"params": {"objective": "tweedie", "tweedie_variance_power": bp["tvp"],
                            "learning_rate": bp["lr"], "num_leaves": bp["nl"],
                            "min_data_in_leaf": bp["md"], "feature_fraction": bp["ff"],
                            "bagging_fraction": bp["bf"], "bagging_freq": 1,
                            "lambda_l1": bp["l1"], "lambda_l2": bp["l2"],
                            "max_bin": PARAM["max_bin"]},
                 "n": bp["n"], "wape_optuna": st.best_value}
    else:
        MEJOR = {"params": {"objective": "tweedie", "tweedie_variance_power": 1.22,
                            "learning_rate": 0.024, "num_leaves": 22, "min_data_in_leaf": 34,
                            "feature_fraction": 0.8, "bagging_fraction": 0.85, "bagging_freq": 1,
                            "max_bin": PARAM["max_bin"]}, "n": 293, "wape_optuna": None}
    CK.write_text(json.dumps(MEJOR, indent=2, default=str))
    print(f"{PARAM['n_trials']} trials en {(time.time() - t0) / 60:.1f} min")

P, NB = MEJOR["params"], MEJOR["n"]
print(f"mejor en febreros: {MEJOR['wape_optuna']}")
print(f"  arboles {NB} | lr {P['learning_rate']:.3f} | hojas {P['num_leaves']} | "
      f"min_data {P['min_data_in_leaf']} | tvp {P['tweedie_variance_power']:.2f}")

## 7 — Reporte y A/B de la segmentación

Se mide mes por mes con las 15 semillas. Y como a este tamaño entrenar es barato, se entrena
**también** el modelo único y se comparan **pareado por fold**: restar el mismo mes cancela su
dificultad propia y deja sólo el efecto de segmentar.

In [ ]:
LIMPIAS = [a for a in PARAM["anclas_reporte"] if a not in PARAM["anclas_optuna"]]
filas = []
for a in PARAM["anclas_reporte"]:
    corte = pm(a, H)
    f = {"ancla": a, "origen": corte}
    ps = prediccion("segmentado", corte, P, NB, PARAM["semillas"]) if PARAM["segmentar"] else None
    pu = (prediccion("unico", corte, P, NB, PARAM["semillas"])
          if PARAM["comparar_sin_segmentar"] or not PARAM["segmentar"] else None)
    if ps is not None:
        f["segmentado"] = wape(REAL[a], ps)
        f["tn_seg"] = float(ps.sum())
    if pu is not None:
        f["unico"] = wape(REAL[a], pu)
    f["real_tn"] = float(REAL[a].dropna().sum())
    if "tn_seg" in f and f["tn_seg"] > 0:
        f["real/pred"] = f["real_tn"] / f["tn_seg"]
    filas.append(f)
    print(f"  ancla {a} (origen {corte}): " + "  ".join(
        f"{k} {f[k]:.4f}" for k in ("segmentado", "unico") if k in f))

rep = pd.DataFrame(filas)
print("\n" + rep.round(4).to_string(index=False))
print(f"\nmedia 5 folds: " + "  ".join(
    f"{k} {rep[k].mean():.4f}" for k in ("segmentado", "unico") if k in rep))
print(f"meses que Optuna NO vio {LIMPIAS}: " + "  ".join(
    f"{k} {rep[rep.ancla.isin(LIMPIAS)][k].mean():.4f}"
    for k in ("segmentado", "unico") if k in rep))

# Sesgo de nivel: ¿el modelo predice el tonelaje total o se queda corto?
if "real/pred" in rep.columns:
    r = rep["real/pred"]
    print(f"\nratio real/predicho por fold: {[round(x, 3) for x in r]}")
    print(f"  mediana {r.median():.3f} | rango {r.min():.3f}-{r.max():.3f} | sd {r.std():.3f}")
    if r.median() > 1.03:
        print(f"  >>> SUB-PREDICE el agregado en la mediana de los folds. Es el mismo sintoma "
              f"que tuvo el z705.")
    if r.std() > 0.08:
        print(f"  >>> Y la dispersion entre meses es ALTA: un multiplicador global no esta "
              f"determinado.\n  >>> (recordatorio: el multiplicador perdio 7 de 7 veces en el LB)")

if {"segmentado", "unico"} <= set(rep.columns):
    d = rep["segmentado"] - rep["unico"]
    ee = d.std(ddof=1) / np.sqrt(len(d))
    print(f"\nsegmentar vs unico, diferencia pareada: {d.mean():+.4f} +- {ee:.4f} (error est.)"
          f" | gana en {(d < 0).sum()}/{len(d)} folds")
    print("  " + ("REAL" if abs(d.mean()) > 2 * ee else
                  "dentro del ruido: la mejora es chica, no un cambio de regimen"))

## 8 — Inferencia y submits

Se entrena con toda la historia hasta 201910 —la última cuya clase ya ocurrió— y se predice
202002 desde las filas de 201912. Se generan el segmentado, el único y el promedio de ambos:
a este costo, tener las tres es gratis y el promedio suele caer por debajo de las dos.

In [ ]:
CORTE_INF = PARAM["periodo_inferencia"]
SUB = {}
if PARAM["segmentar"]:
    SUB["segmentado"] = prediccion("segmentado", CORTE_INF, P, NB, PARAM["semillas"])
if PARAM["comparar_sin_segmentar"] or not PARAM["segmentar"]:
    SUB["unico"] = prediccion("unico", CORTE_INF, P, NB, PARAM["semillas"])
if len(SUB) == 2:
    SUB["promedio"] = (SUB["segmentado"] + SUB["unico"]) / 2

lo, hi = PARAM["banda_nivel"]
oficiales = sorted(TARGET)
SUBMITS = []
for nom, s in SUB.items():
    v = s.reindex(oficiales).fillna(0.0).clip(lower=0.0)
    t = v.sum()
    est = "dentro" if lo <= t <= hi else ("ALTO" if t > hi else "BAJO")
    f = DIR_OUT / f"z713_{nom}.csv"
    pd.DataFrame({"product_id": v.index, "tn": v.values}).to_csv(f, index=False)
    SUBMITS.append((f, f"z713 producto {nom}, top{PARAM['n_top']}, "
                       f"{len(FEATS)} feats, {len(PARAM['semillas'])} semillas, "
                       f"{'crudo' if not PARAM['escalar'] else 'escalado'}"))
    print(f"  {nom:11s} {t:9,.0f} tn  [{est} de la banda {lo:,}-{hi:,}]  faltantes: "
          f"{int((v == 0).sum())}  -> {f.name}")

(DIR_EXP / "resumen.json").write_text(json.dumps(
    {"mejor": MEJOR, "reporte": filas, "features": FEATS,
     "totales": {k: float(s.reindex(oficiales).fillna(0).sum()) for k, s in SUB.items()},
     "param": {k: v for k, v in PARAM.items() if k != "semillas"}}, indent=2, default=str))


def kaggle_submit(f: Path, msg: str):
    flag = f.with_suffix(".done")
    if flag.exists():
        print("ya subido:", f.name); return
    r = subprocess.run(["kaggle", "competitions", "submit", "-c", PARAM["kaggle_competition"],
                        "-f", str(f), "-m", msg], capture_output=True, text=True)
    print(f.name, "->", (r.stdout or r.stderr).strip()[:160])
    if r.returncode == 0:
        flag.write_text(msg)


if PARAM["submit"]:
    for f, msg in SUBMITS:
        kaggle_submit(f, msg)
else:
    print("submit=False -> CSVs en", DIR_OUT)